In [1]:
from openai import OpenAI

In [ ]:
model="gpt-4.1-mini"

In [ ]:
client=OpenAI(
    api_key="api"
)


In [ ]:
import tiktoken

def num_tokens_from_messages(messages, model="gpt-4o-mini-2024-07-18"):
    """Return the number of tokens used by a list of messages."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        print("Warning: model not found. Using o200k_base encoding.")
        encoding = tiktoken.get_encoding("o200k_base")

    if model in {
        "gpt-3.5-turbo-0125",
        "gpt-4-0314",
        "gpt-4-32k-0314",
        "gpt-4-0613",
        "gpt-4-32k-0613",
        "gpt-4o-mini-2024-07-18",
        "gpt-4o-2024-08-06"
        }:
        tokens_per_message = 3
        tokens_per_name = 1
    elif "gpt-3.5-turbo" in model:
        print("Warning: gpt-3.5-turbo may update over time. Returning num tokens assuming gpt-3.5-turbo-0125.")
        return num_tokens_from_messages(messages, model="gpt-3.5-turbo-0125")
    elif "gpt-4o-mini" in model:
        print("Warning: gpt-4o-mini may update over time. Returning num tokens assuming gpt-4o-mini-2024-07-18.")
        return num_tokens_from_messages(messages, model="gpt-4o-mini-2024-07-18")
    elif "gpt-4o" in model:
        print("Warning: gpt-4o and gpt-4o-mini may update over time. Returning num tokens assuming gpt-4o-2024-08-06.")
        return num_tokens_from_messages(messages, model="gpt-4o-2024-08-06")
    elif "gpt-4" in model:
        print("Warning: gpt-4 may update over time. Returning num tokens assuming gpt-4-0613.")
        return num_tokens_from_messages(messages, model="gpt-4-0613")
    else:
        raise NotImplementedError(f"num_tokens_from_messages() is not implemented for model {model}.")

    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3
    return num_tokens

In [ ]:
ai_article_headings = [
    "I. Introduction A. Definition of Artificial Intelligence B. Brief History of AI C. Importance of Understanding AI",
    "II. Types of AI A. Narrow AI B. General AI C. Superintelligent AI",
    "III. Key Technologies in AI A. Machine Learning B. Deep Learning C. Natural Language Processing (NLP) D. Computer Vision",
    "IV. Applications of AI A. Healthcare B. Finance C. Education D. Transportation E. Entertainment",
    "V. AI in Everyday Life A. Virtual Assistants B. Recommendation Systems C. Smart Homes D. Autonomous Vehicles",
    "VI. Ethical Considerations A. Bias in AI B. Privacy Concerns C. Job Displacement D. Accountability",
    "VII. AI and Society A. Impact on Economy B. Changes in Workforce C. Social Implications",
    "VIII. AI Research and Development A. Major AI Research Labs B. Breakthroughs and Innovations C. Future Trends",
    "IX. AI in Business A. AI-driven Decision Making B. Automation of Tasks C. Competitive Advantages",
    "X. AI Governance and Policy A. Government Regulations B. International Standards C. Ethical Guidelines",
    "XI. AI Safety and Risks A. AI Alignment B. Security Threats C. Long-term Risks",
    "XII. AI and Creativity A. Generative AI B. AI in Art and Music C. AI in Writing and Content Creation",
    "XIII. Human-AI Collaboration A. Augmenting Human Abilities B. Hybrid Teams C. Human-in-the-loop Systems",
    "XIV. AI and Education A. Personalized Learning B. Intelligent Tutoring Systems C. AI in Research",
    "XV. AI in Healthcare A. Diagnostics B. Drug Discovery C. Patient Care",
    "XVI. AI and Robotics A. Industrial Robots B. Service Robots C. Autonomous Systems",
    "XVII. Challenges in AI A. Data Limitations B. Computational Constraints C. Interpretability and Explainability",
    "XVIII. Future of AI A. Emerging Trends B. Potential Breakthroughs C. Speculations on AGI",
    "XIX. Glossary A. Key Terms B. Definitions",
    "XX. References A. List of Sources B. Further Reading C. Suggested Research Papers",
]

In [ ]:
system_prompt = "You are a helpful assistant for an AI Professor. You are writing a series of articles about Artificial Intelligence. You have been given a list of headings for each article. You need to write a short paragraph for each heading. You can use the headings as a starting point for your writing.\n\n"
system_prompt+="all of the  subheadings:\n"

messages=[]
for heading in ai_article_headings:
    system_prompt+=f"{heading}\n"

messages.append({"role":"system","content":system_prompt})
max_token_size=2048

In [ ]:
for heading in ai_article_headings:
    messages.append({"role": "user", "content": f"Write a very large paragraph about {heading}. Make it very long and detailed."})


    responce=client.responses.create(
        model=model,
        input=messages,
        store=False
    )
    messages.append({"role": "assistant", "content": response.output_text})
    print("Current message count", len(messages))
    print("Current token count", num_tokens_from_messages(messages))

    while num_tokens_from_messages(messages,model="gpt-4o-mini")>2048:
         non_system_msg_index = next(
            (i for i, msg in enumerate(messages) if msg["role"] not in ["system", "developer"]), None
    if non_system_msg_index is not None:
        messages.pop(non_system_msg_index)
        print("Removed a message to reduce token count!")



In [ ]:
2